# Notebook 3 — NLP Feature Engineering
### Fake Internship & Job Scam Detection System

**Purpose:** Generate structural and NLP-driven features from raw job listing data for downstream fraud detection modelling.  
All features are engineered without leaking the target label (`is_scam`).

---
| Step | Section |
|------|---------|
| 1 | Dataset Loading & Column Preparation |
| 2 | Base Feature Engineering |
| 3 | NLP Feature Generation |
| 4 | Fraud Signal Generation |
| 5 | Dataset Export |

In [1]:
# STEP 1: DATASET LOADING & COLUMN PREPARATION

import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/processed_cleaned_data.csv')

# --- Safe text column initialisation ---
df['description'] = df['description'].fillna('Missing').astype(str)
df['title']       = df['title'].fillna('Missing').astype(str)
df['skills']      = df['skills'].fillna('Missing').astype(str)
df['company']     = df['company'].fillna('Missing').astype(str)
df['work_mode']   = df['work_mode'].fillna('Missing').astype(str)
df['duration']    = df['duration'].fillna('Missing').astype(str)
df['openings']    = df['openings'].fillna('0').astype(str)

# --- Recruiter contact presence flags ---
df['has_website'] = df['company_website'].apply(lambda x: 0 if str(x) == 'Missing' else 1)
df['has_email']   = df['recruiter_email'].apply(lambda x: 0 if str(x) == 'Missing' else 1)

# --- Core text-length proxy metrics (reused downstream) ---
df['skills_count']      = df['skills'].apply(lambda x: 0 if str(x) == 'Missing' else len(str(x).split(',')))
df['title_word_count']  = df['title'].apply(lambda x: len(str(x).split()))
df['desc_char_length']  = df['description'].apply(lambda x: len(str(x)))

print(f"Base data environment successfully prepared. Rows: {df.shape[0]}")

Base data environment successfully prepared. Rows: 13616


In [2]:
# STEP 2: BASE FEATURE ENGINEERING

# --- Openings: extract first numeric token, default to 1 ---
def clean_openings_count(val):
    digits = re.findall(r'\d+', val)
    return int(digits[0]) if digits else 1

df['numeric_openings'] = df['openings'].apply(clean_openings_count)
df['is_bulk_hiring']   = df['numeric_openings'].apply(lambda x: 1 if x >= 25 else 0)

# --- Work-mode & duration flags ---
df['is_wfh']              = df['work_mode'].str.lower().apply(
    lambda x: 1 if 'remote' in x or 'wfh' in x or 'home' in x else 0)
df['is_short_duration']   = df['duration'].str.lower().apply(
    lambda x: 1 if '1 month' in x or 'week' in x else 0)
df['is_wfh_short_duration'] = ((df['is_wfh'] == 1) & (df['is_short_duration'] == 1)).astype(int)

# --- Company name length (unusually long/generic names are a signal) ---
df['company_name_word_count'] = df['company'].apply(lambda x: len(str(x).split()))

# --- Embedded URL count in description ---
url_pattern = r'https?://\S+|www\.\S+'
df['link_count_desc'] = df['description'].apply(
    lambda x: len(re.findall(url_pattern, x, flags=re.IGNORECASE)))

# --- Phone-like number count in description ---
phone_pattern = r'\b\d{7,12}\b'
df['contact_count_desc'] = df['description'].apply(
    lambda x: len(re.findall(phone_pattern, x)))

# --- ALL-CAPS ratio (spammy listings over-capitalise) ---
def calculate_caps_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0

df['all_caps_ratio_desc'] = df['description'].apply(calculate_caps_ratio)

# --- Legacy scam keyword count (binary presence, unweighted) ---
scam_keywords = [
    'registration fee', 'security deposit', 'processing fee', 'training fee',
    'training charge', 'whatsapp directly', 'investment', 'earn online',
    'pay to work', 'income source', 'telegram link', 'typing job', 'form filling'
]
desc_lower_series   = df['description'].str.lower()
df['keyword_count'] = desc_lower_series.apply(
    lambda text: sum(1 for word in scam_keywords if word in text))

# --- Binary urgency flag (legacy; replaced by urgency_score in Step 4) ---
df['has_urgency_words'] = desc_lower_series.str.contains(
    'immediately|urgent|limited|hurry', regex=True).astype(int)

# --- Composite ratio features ---
df['stipend_to_skills_ratio'] = df['cleaned_stipend_monthly'] / (df['skills_count'] + 1)
df['text_density_index']      = df['title_word_count'] / (df['desc_char_length'] + 1)
df['low_skills_high_urgency'] = ((df['skills_count'] <= 1) & (df['has_urgency_words'] == 1)).astype(int)
df['is_anonymous_recruiter']  = ((df['has_website'] == 0) & (df['has_email'] == 0)).astype(int)

print("All structural features across all dataset columns compiled perfectly with zero data leakage!")

All structural features across all dataset columns compiled perfectly with zero data leakage!


In [3]:

# STEP 3: NLP FEATURE GENERATION

try:
    from textstat import flesch_reading_ease
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'textstat', '-q'])
    from textstat import flesch_reading_ease

# --- Flesch Reading Ease Score ---

df['readability_score'] = df['description'].apply(flesch_reading_ease)

print("STEP 3 complete — NLP readability features generated.")
print(f"  readability_score  | mean: {df['readability_score'].mean():.2f}  "
      f"min: {df['readability_score'].min():.2f}  max: {df['readability_score'].max():.2f}")

STEP 3 complete — NLP readability features generated.
  readability_score  | mean: 16.49  min: -433.97  max: 100.24


In [5]:

# STEP 4: FRAUD SIGNAL GENERATION

# Features added:
#   fraud_phrase_score  — Weighted sum of known fraud phrases
#   urgency_score       — Count of urgency-language tokens
#   contact_risk_score  — Weighted sum of risky contact cues
#   skill_richness_score — Ratio of skills to title length



# 4-A  fraud_phrase_score

FRAUD_PHRASE_WEIGHTS = {
    'registration fee' : 10,
    'security deposit' : 10,
    'investment'       : 10,
    'pay to work'      : 10,
    'processing fee'   :  8,
    'training fee'     :  8,
    'earn online'      :  8,
    'telegram'         :  8,
    'whatsapp'         :  6,
    'form filling'     :  6,
    'typing job'       :  6,
    'income source'    :  5,
    'work from home job':  5,
    'daily payment'    :  7,
    'no experience needed': 4,
}

def compute_fraud_phrase_score(text: str) -> int:
    """Return cumulative weight score of fraud phrases present in text."""
    text_lower = text.lower()
    return sum(weight for phrase, weight in FRAUD_PHRASE_WEIGHTS.items()
               if phrase in text_lower)

df['fraud_phrase_score'] = df['description'].apply(compute_fraud_phrase_score)


# 4-B  urgency_score

URGENCY_PHRASES = [
    'urgent', 'immediately', 'limited seats', 'limited openings',
    'hurry', 'apply now', 'last date', 'closing soon',
    'do not miss', 'act fast', 'fast track', 'first come first'
]

def compute_urgency_score(text: str) -> int:
    """Count distinct urgency phrases found in the description."""
    text_lower = text.lower()
    return sum(1 for phrase in URGENCY_PHRASES if phrase in text_lower)

df['urgency_score'] = df['description'].apply(compute_urgency_score)

# 4-C  contact_risk_score

CONTACT_RISK_WEIGHTS = {
    'whatsapp'   : 8,
    'telegram'   : 8,
    'gmail.com'  : 5,
    'yahoo.com'  : 5,
    'call now'   : 6,
    'call us'    : 4,
    'contact us on': 4,
    'direct message': 5,
    'dm us'      : 5,
    'ping us'    : 4,
    'chat with'  : 3,
}

def compute_contact_risk_score(text: str) -> int:
    """Return cumulative weight of risky contact-channel cues in text."""
    text_lower = text.lower()
    return sum(weight for cue, weight in CONTACT_RISK_WEIGHTS.items()
               if cue in text_lower)

df['contact_risk_score'] = df['description'].apply(compute_contact_risk_score)

# 4-D  skill_richness_score


df['skill_richness_score'] = df['skills_count'] / (df['title_word_count'] + 1)

print("STEP 4 complete — Fraud signal features generated.")
for feat in ['fraud_phrase_score', 'urgency_score', 'contact_risk_score', 'skill_richness_score']:
    print(f"  {feat:<25} | mean: {df[feat].mean():.4f}  "
          f"non-zero rows: {(df[feat] > 0).sum()}")

STEP 4 complete — Fraud signal features generated.
  fraud_phrase_score        | mean: 1.7815  non-zero rows: 2441
  urgency_score             | mean: 0.2496  non-zero rows: 2615
  contact_risk_score        | mean: 0.9241  non-zero rows: 1618
  skill_richness_score      | mean: 0.7936  non-zero rows: 7570


In [6]:

# STEP 5: DATASET EXPORT

ultimate_feature_lineup = [
    # --- Target ---
    'is_scam',

    # --- Original 18 feature columns (unchanged) ---
    'cleaned_stipend_monthly',
    'has_website',
    'has_email',
    'skills_count',
    'title_word_count',
    'desc_char_length',
    'numeric_openings',
    'is_bulk_hiring',
    'is_wfh_short_duration',
    'company_name_word_count',
    'link_count_desc',
    'contact_count_desc',
    'all_caps_ratio_desc',
    'keyword_count',
    'stipend_to_skills_ratio',
    'text_density_index',
    'low_skills_high_urgency',
    'is_anonymous_recruiter',
    'readability_score',      # Flesch Reading Ease of description
    'fraud_phrase_score',     # Weighted sum of high-risk fraud phrases
    'urgency_score',          # Count of urgency-language cues
    'contact_risk_score',     # Weighted score of informal contact channels
    'skill_richness_score',   # skills_count / (title_word_count + 1)
]

# --- Slice and clean the final dataset ---
model_ready_dataset = df[ultimate_feature_lineup].copy()
model_ready_dataset = model_ready_dataset.fillna(0)   # Safety sweep for any residual NaNs

# --- Verification report ---
print("--- DEFINITIVE MODEL-READY GRID VERIFICATION ---")
print(f"Total Available Data Rows : {model_ready_dataset.shape[0]}")
print(f"Total Unique Model Features: {model_ready_dataset.shape[1]}")
print(f"Remaining NaN values       : {model_ready_dataset.isna().sum().sum()}")
print("\nFinal clean numeric feature lineup for Model Training:")
print(list(model_ready_dataset.columns))

--- DEFINITIVE MODEL-READY GRID VERIFICATION ---
Total Available Data Rows : 13616
Total Unique Model Features: 24
Remaining NaN values       : 0

Final clean numeric feature lineup for Model Training:
['is_scam', 'cleaned_stipend_monthly', 'has_website', 'has_email', 'skills_count', 'title_word_count', 'desc_char_length', 'numeric_openings', 'is_bulk_hiring', 'is_wfh_short_duration', 'company_name_word_count', 'link_count_desc', 'contact_count_desc', 'all_caps_ratio_desc', 'keyword_count', 'stipend_to_skills_ratio', 'text_density_index', 'low_skills_high_urgency', 'is_anonymous_recruiter', 'readability_score', 'fraud_phrase_score', 'urgency_score', 'contact_risk_score', 'skill_richness_score']


In [7]:
# --- Save final model-ready dataset to disk ---
model_ready_dataset.to_csv('../data/model_ready_dataset.csv', index=False)
print("\nFinal model ready dataset saved to '../data/model_ready_dataset.csv'")
print(f"Shape: {model_ready_dataset.shape[0]} rows × {model_ready_dataset.shape[1]} columns")


Final model ready dataset saved to '../data/model_ready_dataset.csv'
Shape: 13616 rows × 24 columns
